In [17]:
import os
import pickle
import numpy as np
import copy
from coffea.util import load
from coffea.lookup_tools.dense_lookup import dense_lookup

# 1. Define the base paths where your files live
base_dir = "/cms/data/jsamudio/boosted/boostedttX/configs/SemiLeptonic_2024/BtagEff"
folders = ["Bkg", "Minor", "QCD", "Sig", "VJets"]

# 2. Dictionaries to hold the raw accumulated histograms
accumulated_all = {}
accumulated_pass = {}

print("Starting b-tag efficiency accumulation...")

# ==========================================
# STEP 1: Accumulate all raw histograms by Sample Split
# ==========================================
for folder in folders:
    filepath = os.path.join(base_dir, folder, "output_all.coffea")
    
    if not os.path.exists(filepath):
        print(f"⚠️ WARNING: File not found -> {filepath}")
        continue
        
    print(f"\n--- Loading {folder} ---")
    try:
        result = load(filepath)
    except Exception as e:
        print(f"❌ ERROR: Could not load {filepath}. ({e})")
        continue

    if 'variables' not in result or 'eff_all' not in result['variables']:
        print(f"⚠️ WARNING: 'eff_all' not found in {folder}. Skipping.")
        continue
        
    h_all = result['variables']['eff_all']
    h_pass = result['variables']['eff_pass']

    for raw_sample_name in h_all.keys():
        
        # ==========================================
        # THE FIX: STRIP THE SUBSAMPLE TAG
        # This forces 'TTToSemiLeptonic__tt+LF', '__tt+C', and '__tt+B' 
        # to all collapse into the master 'TTToSemiLeptonic' key.
        # ==========================================
        master_sample_name = raw_sample_name.split("__")[0]
        
        for dataset_name in h_all[raw_sample_name].keys():
            try:
                # Slice the 5D histogram to 3D (pT, abseta, flavor)
                hist_all_3d = h_all[raw_sample_name][dataset_name]["baseline", "nominal", ...]
                hist_pass_3d = h_pass[raw_sample_name][dataset_name]["baseline", "nominal", ...]
                
                # Accumulate based on the STRIPPED master_sample_name
                if master_sample_name not in accumulated_all:
                    accumulated_all[master_sample_name] = copy.deepcopy(hist_all_3d)
                    accumulated_pass[master_sample_name] = copy.deepcopy(hist_pass_3d)
                    print(f"  [+] Initialized: {master_sample_name} (from {raw_sample_name} / {dataset_name})")
                else:
                    # Sum the histograms together across different physical datasets AND subsamples
                    accumulated_all[master_sample_name] += hist_all_3d
                    accumulated_pass[master_sample_name] += hist_pass_3d
                    print(f"  [➔] Added chunk to: {master_sample_name} (from {raw_sample_name} / {dataset_name})")
                    
            except KeyError as e:
                print(f"  ❌ KeyError for {raw_sample_name}/{dataset_name}: {e}")

# ==========================================
# STEP 2: Divide and create lookup tools
# ==========================================
print("\n--- Calculating Final Efficiencies ---")
efficiency_lookups = {}

for sample_name in accumulated_all.keys():
    
    # Extract the summed raw numpy arrays
    num = accumulated_pass[sample_name].values()
    den = accumulated_all[sample_name].values()
    
    # Calculate efficiency safely (default to 1.0 if denominator is 0)
    eff = np.divide(num, den, out=np.ones_like(num), where=(den > 0))
    
    # Extract the bin edges (Order: pT, abseta, flavor)
    edges = (
        accumulated_all[sample_name].axes[0].edges,
        accumulated_all[sample_name].axes[1].edges,
        accumulated_all[sample_name].axes[2].edges
    )
    
    # Create the lookup tool and store it under the outer sample_name
    lookup = dense_lookup(eff, edges)
    efficiency_lookups[sample_name] = lookup
    print(f"  ✓ Mapped: {sample_name}")

# ==========================================
# STEP 3: Save to Pickle
# ==========================================
output_file = os.path.join(base_dir, "semileptonic_btag_eff_2024.pkl")

with open(output_file, "wb") as fout:
    pickle.dump(efficiency_lookups, fout)
    
print("\n" + "="*50)
print(f"✅ Successfully compiled {len(efficiency_lookups)} unique efficiency maps!")
print(f"💾 Saved to: {output_file}")
print("="*50)

Starting b-tag efficiency accumulation...

--- Loading Bkg ---
  [+] Initialized: TTToSemiLeptonic (from TTToSemiLeptonic / TTtoLNu2Q_2024)
  [+] Initialized: TTTo2L2Nu (from TTTo2L2Nu / TTto2L2Nu_2024)
  [+] Initialized: TTToHadronic (from TTToHadronic / TTto4Q_2024)
  [+] Initialized: TTbb_SemiLeptonic (from TTbb_SemiLeptonic / TTBBtoLNu2Q_2024)
  [+] Initialized: TTbb_2L2Nu (from TTbb_2L2Nu / TTBBto2L2Nu_2024)
  [+] Initialized: TTbb_Hadronic (from TTbb_Hadronic / TTBBto4Q_2024)

--- Loading Minor ---
  [+] Initialized: TTX (from TTX / TZQ_ll_2024)
  [➔] Added chunk to: TTX (from TTX / THQ_2024)
  [➔] Added chunk to: TTX (from TTX / THW_2024)
  [➔] Added chunk to: TTX (from TTX / TTTT_2024)
  [➔] Added chunk to: TTX (from TTX / TTWJets_2024)
  [+] Initialized: SingleTop (from SingleTop / ST_antitop_t_LNu_2024)
  [➔] Added chunk to: SingleTop (from SingleTop / ST_antitop_t_2Q_2024)
  [➔] Added chunk to: SingleTop (from SingleTop / ST_top_t_LNu_2024)
  [➔] Added chunk to: SingleTop (f

In [18]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages

# ==========================================
# 1. CONFIGURATION
# ==========================================
hep.style.use(hep.style.CMS)

# Update this path to wherever your pkl file lives
PKL_PATH = "/cms/data/jsamudio/boosted/boostedttX/configs/SemiLeptonic_2024/BtagEff/semileptonic_btag_eff_2024.pkl"
OUTPUT_PDF = "BTag_MC_Efficiencies_2024.pdf"

# We will plot the efficiency for the Barrel region (|eta| = 0.5)
# You can change this to 1.5 to check the Endcap region!
ETA_EVAL = 0.5 

# ==========================================
# 2. LOAD MAPS & PLOT
# ==========================================
print(f"Loading efficiency maps from {PKL_PATH}...")
with open(PKL_PATH, "rb") as f:
    eff_maps = pickle.load(f)

print(f"Found {len(eff_maps)} samples. Generating plots...")

with PdfPages(OUTPUT_PDF) as pdf:
    for sample_name, lookup in eff_maps.items():
        
        # 1. Extract the pT bin edges directly from the lookup tool's internal axes
        # (Axis 0 is pT, Axis 1 is abseta, Axis 2 is flavor)
        try:
            pt_edges = lookup._axes[0]
            pt_centers = 0.5 * (pt_edges[:-1] + pt_edges[1:])
        except AttributeError:
            print(f"⚠️ Skipping {sample_name}: Could not parse dense_lookup axes.")
            continue
            
        # 2. Create uniform arrays to feed into the lookup function
        eta_arr = np.full_like(pt_centers, ETA_EVAL)
        
        # 3. Query the lookup tool for the 3 flavors
        eff_b = lookup(pt_centers, eta_arr, np.full_like(pt_centers, 5))
        eff_c = lookup(pt_centers, eta_arr, np.full_like(pt_centers, 4))
        eff_l = lookup(pt_centers, eta_arr, np.full_like(pt_centers, 0))
        
        # 4. Plotting
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # We use hep.histplot because it naturally draws the step functions across the bin edges
        hep.histplot(eff_b, bins=pt_edges, ax=ax, label=r'True $b$-jets (flavour = 5)', color='#1f77b4', linewidth=2)
        hep.histplot(eff_c, bins=pt_edges, ax=ax, label=r'True $c$-jets (flavour = 4)', color='#ff7f0e', linewidth=2)
        hep.histplot(eff_l, bins=pt_edges, ax=ax, label=r'Light/Gluon (flavour = 0)', color='#2ca02c', linewidth=2)
        
        # Styling
        ax.set_title(f"Sample: {sample_name}", fontsize=14, pad=10)
        ax.set_xlabel(r"Jet $p_{T}$ [GeV]")
        ax.set_ylabel(r"MC B-Tagging Efficiency ($\epsilon$)")
        
        # The light jet efficiency will be tiny (e.g., 0.01), and b-jets will be huge (e.g., 0.80)
        # A log scale on the Y-axis is standard so you can actually see the light jet shape!
        ax.set_yscale('log')
        ax.set_ylim(1e-3, 1.5) 
        ax.set_xlim(pt_edges[0], pt_edges[-1])
        
        ax.legend(loc='upper right', fontsize=12)
        ax.grid(axis='y', which='both', linestyle='--', alpha=0.3)
        hep.cms.label("Preliminary", data=False, lumi=109.9, ax=ax, com=13.6)
        
        # Save to the PDF
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f"\n✅ Successfully generated {len(eff_maps)} plots!")
print(f"💾 Saved to: {OUTPUT_PDF}")

Loading efficiency maps from /cms/data/jsamudio/boosted/boostedttX/configs/SemiLeptonic_2024/BtagEff/semileptonic_btag_eff_2024.pkl...
Found 17 samples. Generating plots...

✅ Successfully generated 17 plots!
💾 Saved to: BTag_MC_Efficiencies_2024.pdf


In [16]:
import pickle
import numpy as np

# Update this path if needed
PKL_PATH = "/cms/data/jsamudio/boosted/boostedttX/configs/SemiLeptonic_2024/BtagEff/semileptonic_btag_eff_2024.pkl"

print(f"Loading {PKL_PATH}...")
with open(PKL_PATH, "rb") as f:
    eff_maps = pickle.load(f)

# Define a standard kinematic test point: pt = 100 GeV, |eta| = 0.5
test_pt = np.array([100.0])
test_eta = np.array([0.5])

print("\n--- Numerical Efficiency Check (pT=100, |eta|=0.5) ---")
print(f"{'Sample Name':<35} | {'b-jet (5)':<10} | {'c-jet (4)':<10} | {'light (0)':<10}")
print("-" * 74)

for sample_name, lookup in eff_maps.items():
    # Query the lookup maps for the three flavors
    try:
        eff_b = lookup(test_pt, test_eta, np.array([5]))[0]
        eff_c = lookup(test_pt, test_eta, np.array([4]))[0]
        eff_l = lookup(test_pt, test_eta, np.array([0]))[0]
        
        print(f"{sample_name:<35} | {eff_b:<10.4f} | {eff_c:<10.4f} | {eff_l:<10.4f}")
    except Exception as e:
        print(f"{sample_name:<35} | ERROR querying map: {e}")

print("-" * 74)

Loading /cms/data/jsamudio/boosted/boostedttX/configs/SemiLeptonic_2024/BtagEff/semileptonic_btag_eff_2024.pkl...

--- Numerical Efficiency Check (pT=100, |eta|=0.5) ---
Sample Name                         | b-jet (5)  | c-jet (4)  | light (0) 
--------------------------------------------------------------------------
TTToSemiLeptonic                    | 0.9368     | 0.1114     | 0.0110    
TTTo2L2Nu                           | 0.9306     | 0.1606     | 0.0241    
TTToHadronic                        | 0.9593     | 0.0855     | 0.0111    
TTbb_SemiLeptonic                   | 0.9140     | 0.1082     | 0.0085    
TTbb_2L2Nu                          | 0.9144     | 0.1667     | 0.0206    
TTbb_Hadronic                       | 1.0000     | 0.1000     | 0.0000    
TTX                                 | 0.9153     | 0.0934     | 0.0128    
SingleTop                           | 0.9326     | 0.1225     | 0.0140    
VV                                  | 0.9590     | 0.1303     | 0.0000    
QCD  